In [1]:
import pandas as pd

# Load the clean data we saved in Step 32
df = pd.read_csv("../data/processed/jobs_clean.csv")

print("Rows and columns:", df.shape)
df[["title", "role", "city", "description"]].head()

Rows and columns: (2314, 23)


,title,role,city,description
0,Data Analyst,Data Analyst,India (unspecified),Company Description Ivtex Corporate Solutions ...
1,Data Analyst,Data Analyst,Karnataka,We are looking for a Data Analyst who can coll...
2,Data Analyst,Data Analyst,India (unspecified),About the Role : We are looking for a highly a...
3,Data Analyst,Data Analyst,Pune,"This job is with Cornerstone OnDemand, an incl..."
4,Data Analyst,Data Analyst,India (unspecified),Job Title: Data Analyst Location: India (Remot...


In [2]:
# Step A: clean up the description text so words are easy to find
def clean_text(text):
    t = str(text).lower()
    for symbol in [".", ",", "/", "(", ")", "-", "_", ":", ";", "!", "\n"]:
        t = t.replace(symbol, " ")
    return " " + t + " "

df["desc_clean"] = df["description"].apply(clean_text)

# Step B: the list of skills we want to search for
skills = ["python", "sql", "excel", "power bi", "tableau", "statistics",
          "machine learning", "deep learning", "nlp", "llm", "genai",
          "aws", "azure", "gcp", "spark", "databricks", "snowflake",
          "airflow", "docker", "git", "tensorflow", "pytorch"]

# Step C: count how many jobs mention each skill
results = []
for skill in skills:
    count = df["desc_clean"].str.contains(" " + skill + " ", regex=False).sum()
    results.append([skill, count])

# Step D: turn the results into a table, sorted from most to least common
skill_counts = pd.DataFrame(results, columns=["skill", "jobs"])
skill_counts["percent"] = (skill_counts["jobs"] / len(df) * 100).round(1)
skill_counts = skill_counts.sort_values("jobs", ascending=False)
skill_counts

,skill,jobs,percent
6,machine learning,376,16.2
0,python,264,11.4
1,sql,210,9.1
12,azure,113,4.9
11,aws,96,4.1
15,databricks,78,3.4
9,llm,71,3.1
14,spark,65,2.8
7,deep learning,64,2.8
8,nlp,62,2.7


In [3]:
# Step A: for each skill, make a True/False column: does this job mention it?
for skill in skills:
    df[skill] = df["desc_clean"].str.contains(" " + skill + " ", regex=False)

# Step B: leave out the "Other" jobs (non-data roles)
data_jobs = df[df["role"] != "Other"]

# Step C: for each role, what % of jobs mention each skill?
skills_by_role = data_jobs.groupby("role")[skills].mean() * 100

# Step D: flip the table so skills are rows and roles are columns (easier to read)
skills_by_role.round(1).T

role,AI Engineer,Business Analyst,Data Analyst,Data Engineer,Data Scientist,ML Engineer
python,12.1,0.3,10.9,21.0,17.5,8.7
sql,0.9,2.6,20.0,21.8,5.8,3.9
excel,0.0,2.9,8.4,0.0,0.3,2.4
power bi,0.0,1.1,7.7,2.1,0.6,0.0
tableau,0.0,0.9,5.7,0.3,1.3,0.0
statistics,0.2,0.9,2.0,0.3,7.4,0.9
machine learning,13.7,0.6,1.0,0.8,34.3,56.7
deep learning,3.0,0.0,0.0,0.0,6.8,8.4
nlp,3.5,0.0,0.0,0.3,8.1,6.0
llm,11.8,0.0,0.0,0.3,2.3,3.6


In [5]:
# Business-side skills
ba_skills = ["stakeholder", "requirement", "agile", "jira", "scrum",
             "documentation", "communication", "sdlc"]

# Same idea as before, but with a space only at the START,
# so plural forms like "stakeholders" and "requirements" also count
for skill in ba_skills:
    df[skill] = df["desc_clean"].str.contains(" " + skill, regex=False)

# Recreate the data-jobs table so it includes the new columns
data_jobs = df[df["role"] != "Other"]

# % of jobs in each role that mention each business skill
ba_by_role = data_jobs.groupby("role")[ba_skills].mean() * 100
ba_by_role.round(1).T

role,AI Engineer,Business Analyst,Data Analyst,Data Engineer,Data Scientist,ML Engineer
stakeholder,2.1,25.6,10.4,4.7,5.8,1.5
requirement,4.9,50.6,11.4,8.7,7.1,4.5
agile,0.0,6.0,0.2,0.8,0.6,2.1
jira,0.0,1.7,0.0,0.0,0.0,0.0
scrum,0.0,2.3,0.2,0.3,0.3,0.3
documentation,0.5,8.6,0.7,0.3,1.0,0.0
communication,1.2,10.3,5.9,1.0,1.6,1.8
sdlc,0.0,1.1,0.2,0.3,0.0,0.0


In [6]:
# All skills, minus "requirement" (too ambiguous, as discussed)
all_skills = skills + ba_skills
all_skills.remove("requirement")

# Turn the many True/False columns into two columns: skill and mentioned
skill_long = df.melt(id_vars=["job_id"], value_vars=all_skills,
                     var_name="skill", value_name="mentioned")

# Keep only the rows where the skill was actually mentioned
skill_long = skill_long[skill_long["mentioned"] == True]
skill_long = skill_long[["job_id", "skill"]]

# Save it
skill_long.to_csv("../data/processed/job_skills.csv", index=False)
print("Saved! Job-skill pairs:", len(skill_long))
skill_long.head()

Saved! Job-skill pairs: 2157


,job_id,skill
12,5837139892,python
34,5876871774,python
40,5867263507,python
45,5896613632,python
57,5891226525,python
